# Odin MIDI-LLM Training (Colab)**一鍵訓練 —— 用情緒標籤數據集**Runtime → Change runtime type → GPU (T4)

In [ ]:
!nvidia-smi -Limport torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
!pip -q install peft accelerate huggingface_hub kaggle 2>&1 | tail -1

In [ ]:
import osos.makedirs("/root/.kaggle", exist_ok=True)open("/root/.kaggle/access_token","w").write('KGAT_7c5ed92d076f80583ea46526ce80f463')os.chmod("/root/.kaggle/access_token", 0o600)print("kaggle token set")

In [ ]:
import osfrom huggingface_hub import snapshot_downloadM="/content/midi_llm"; os.makedirs(M, exist_ok=True)if not os.path.exists(os.path.join(M,"model.safetensors")):    snapshot_download("slseanwu/MIDI-LLM_Llama-3.2-1B", local_dir=M)print(sorted(os.listdir(M))[:6])

In [ ]:
!kaggle datasets download -d dgdmmmmmmmmmmm/odin-mood-dataset -p /content --unzip!ls -la /content/*.jsonl!wc -l /content/mood_train.jsonl

In [ ]:
open("/content/train.py","w").write(r'''import sys, json, os, torchsys.stdout.reconfigure(encoding="utf-8", errors="replace")from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModelingfrom peft import LoraConfig, get_peft_modelMODEL="/content/midi_llm"MAXLEN=768; BATCH=4; ACCUM=2; STEPS=8000; LR=3e-4; RANK=64; CAP=120000tok=AutoTokenizer.from_pretrained(MODEL)if tok.pad_token is None: tok.pad_token=tok.eos_tokenmodel=AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)model.config.use_cache=Falsetry: model.enable_input_require_grads()except Exception: passcfg=LoraConfig(r=RANK, lora_alpha=RANK*2,  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")model=get_peft_model(model,cfg); model.print_trainable_parameters()ex=[]for line in open("/content/mood_train.jsonl", encoding="utf-8"):    line=line.strip()    if not line: continue    r=json.loads(line)    t=tok(r["text"], add_special_tokens=True)["input_ids"]    m=[int(x) for x in r["midi_tokens"]]    ids=(t+m)[:MAXLEN]; lab=([-100]*len(t)+m[:MAXLEN-len(t)])[:MAXLEN]    if len(ids)<20: continue    pad=MAXLEN-len(ids)    ex.append({"input_ids":torch.tensor(ids+[tok.pad_token_id]*pad),               "labels":torch.tensor(lab+[-100]*pad),               "attention_mask":torch.tensor([1]*len(ids)+[0]*pad)})    if len(ex)>=CAP: breakprint("Examples:", len(ex), flush=True)class DS(torch.utils.data.Dataset):    def __init__(s,e): s.e=e    def __len__(s): return len(s.e)    def __getitem__(s,i): return s.e[i]args=TrainingArguments(output_dir="/content/lora_out", per_device_train_batch_size=BATCH,  gradient_accumulation_steps=ACCUM, max_steps=STEPS, learning_rate=LR, bf16=True,  logging_steps=20, save_steps=400, save_total_limit=3, remove_unused_columns=False,  report_to=[], warmup_steps=200, lr_scheduler_type="cosine", dataloader_num_workers=2)tr=Trainer(model=model, args=args, train_dataset=DS(ex),           data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False))tr.train()model.save_pretrained("/content/lora_out/final"); tok.save_pretrained("/content/lora_out/final")print("DONE -> /content/lora_out/final")''')print("train.py written")

In [ ]:
!python -u /content/train.py

In [ ]:
!cd /content && zip -qr odin_lora.zip lora_outfrom google.colab import filesfiles.download("/content/odin_lora.zip")